In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ── paths ──────────────────────────────────────────────────────
SHAPE_LOG       = r"C:/ClaudeCode/research/01. regime_identification/input/shape_log.csv"
ENRICHED_LOG    = r"C:/ClaudeCode/research/03. validation_analysis/shape_log_enriched.csv"
STOCK           = r"C:/ClaudeCode/Raw Data/Stock and Production/FCPO Stock 3Y.xlsx"
PRODUCTION      = r"C:/ClaudeCode/Raw Data/Stock and Production/MPOB Production 3Y.xlsx"
EXPORT          = r"C:/ClaudeCode/Raw Data/Stock and Production/MPOB Export 3Y.xlsx"
USDMYR          = r"C:/ClaudeCode/Raw Data/Variable Analysis Extra Data/FX_IDC_USDMYR, 1D_1227e.csv"
PALMSOY         = r"C:/ClaudeCode/Raw Data/Variable Analysis Extra Data/MYX_DLY_FCPO1!_2_CBOT_DL_ZL1!, 1D_61912.csv"
CRUDE           = r"C:/ClaudeCode/Raw Data/Variable Analysis Extra Data/NYMEX_DL_CL1!, 1D_84001.csv"
ENSO_ASCII      = r"C:/ClaudeCode/Raw Data/ENSO/oni.ascii.txt"
LOG_FILE        = r"C:/ClaudeCode/research/03. validation_analysis/further_validation_log.txt"

# ── shape name mapping ─────────────────────────────────────────
# Internal code uses 0.0/0.1/0.2/1/2
# Display layer converts to readable names
SHAPE_NAMES = {
    '0.0': 'SB  (Steep Backwardation)',
    '0.1': 'MB  (Mild Backwardation)',
    '0.2': 'TB  (Transitional Backwardation)',
    '1':   'C   (Contango)',
    '2':   'F   (Flat/Back-Inversion)',
}
SHAPE_SHORT = {
    '0.0': 'SB', '0.1': 'MB',
    '0.2': 'TB', '1': 'C', '2': 'F',
}

def display_shape(code):
    return SHAPE_NAMES.get(str(code), str(code))

def display_short(code):
    return SHAPE_SHORT.get(str(code), str(code))

# ── log writer ─────────────────────────────────────────────────
def write_log(study_id, content):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
    header = f"""
{"="*70}
{study_id}
Run: {timestamp}
{"="*70}
"""
    with open(LOG_FILE, 'a', encoding='utf-8') as f:
        f.write(header + content + "\n")
    print(f"[LOG] Written -- {study_id}")

# ── train/test split (locked, same as all prior work) ──────────
TRAIN_END  = '2024-12-31'
TEST_START = '2025-01-01'

print("Setup complete.")
print("Shape name mapping:")
for k, v in SHAPE_NAMES.items():
    print(f"  {k} -> {v}")

Setup complete.
Shape name mapping:
  0.0 -> SB  (Steep Backwardation)
  0.1 -> MB  (Mild Backwardation)
  0.2 -> TB  (Transitional Backwardation)
  1 -> C   (Contango)
  2 -> F   (Flat/Back-Inversion)


In [2]:
# ── load shape log (enriched version with duration + prior) ───
df_enriched = pd.read_csv(
    ENRICHED_LOG,
    dtype={'shape': str, 'prior_shape': str},
    parse_dates=['date'])
df_enriched = df_enriched[
    df_enriched['date'] >= '2017-01-01'
].sort_values('date').reset_index(drop=True)

# ── load MPOB ──────────────────────────────────────────────────
def load_mpob(path, col_name):
    df = pd.read_excel(path, parse_dates=True)
    date_col = df.columns[0]
    val_col  = df.columns[1]
    df = df.rename(columns={date_col: 'date', val_col: col_name})
    df['date'] = pd.to_datetime(df['date'])
    df = df.dropna(subset=['date'])
    return df[['date', col_name]].sort_values('date')

stock = load_mpob(STOCK,      'stock_raw')
prod  = load_mpob(PRODUCTION, 'production_raw')
exp   = load_mpob(EXPORT,     'export_raw')

# ── load macro (CSV with Unix timestamps) ─────────────────────
def load_macro(path, col_name):
    df = pd.read_csv(path)
    df['date'] = pd.to_datetime(df['time'], unit='s')
    df = df.rename(columns={'close': col_name})
    return df[['date', col_name]].sort_values('date')

usdmyr  = load_macro(USDMYR,   'usd_myr')
palmsoy = load_macro(PALMSOY,  'palm_soy')
crude   = load_macro(CRUDE,    'crude_oil')

# ── load ENSO (3-month season format) ─────────────────────────
enso = pd.read_csv(ENSO_ASCII, sep=r'\s+',
    names=['seas','year','total','oni'], skiprows=1)
seas_month = {
    'DJF':1, 'JFM':2, 'FMA':3, 'MAM':4,
    'AMJ':5, 'MJJ':6, 'JJA':7, 'JAS':8,
    'ASO':9, 'SON':10, 'OND':11, 'NDJ':12,
}
enso['month'] = enso['seas'].map(seas_month)
enso['date'] = pd.to_datetime(
    enso[['year','month']].assign(day=1))
enso_long = enso[['date','oni']].dropna().sort_values('date')

# ── merge everything onto enriched shape log ───────────────────
df = df_enriched.copy()

for src, col in [(stock,'stock_raw'),
                  (prod, 'production_raw'),
                  (exp,  'export_raw')]:
    df = pd.merge_asof(
        df.sort_values('date'),
        src.sort_values('date'),
        on='date', direction='backward')

df = pd.merge_asof(
    df.sort_values('date'),
    enso_long.sort_values('date'),
    on='date', direction='backward')

for src, col in [(usdmyr,'usd_myr'),
                  (palmsoy,'palm_soy'),
                  (crude,  'crude_oil')]:
    df = pd.merge_asof(
        df.sort_values('date'),
        src.sort_values('date'),
        on='date', direction='backward')

# ── derived variables ──────────────────────────────────────────
df['stock_pct']        = df['stock_raw'].rank(pct=True)
df['prod_mom_1m']      = df['production_raw'].pct_change(21)*100
df['prod_mom_3m']      = df['production_raw'].pct_change(63)*100
df['prod_yoy']         = df['production_raw'].pct_change(252)*100
df['export_yoy']       = df['export_raw'].pct_change(252)*100
df['usd_myr_chg_4w']   = df['usd_myr'].pct_change(20)*100
df['palm_soy_chg_4w']  = df['palm_soy'].pct_change(20)*100
df['crude_chg_4w']     = df['crude_oil'].pct_change(20)*100
df['month']            = df['date'].dt.month

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['prior_shape_enc'] = le.fit_transform(
    df['prior_shape'].fillna('unknown'))

df['stock_state'] = np.where(
    df['stock_pct'] >= df['stock_pct'].median(),
    'High', 'Low')
df['prod_state'] = np.where(
    df['prod_mom_3m'] >= 0, 'Rising', 'Falling')
df['stock_prod_quad'] = df['stock_state'] + '_' + df['prod_state']

quad_map = {
    'High_Falling': 3,
    'Low_Rising':   2,
    'Low_Falling':  1,
    'High_Rising':  0,
}
df['stock_prod_interaction'] = df['stock_prod_quad'].map(
    quad_map).fillna(0)

print(f"Panel built: {len(df)} rows")
print(f"Date range: {df['date'].min().date()} to "
      f"{df['date'].max().date()}")
print(f"\nMissing values in key columns:")
key_cols = ['stock_pct','prod_mom_3m','oni',
            'usd_myr_chg_4w','palm_soy_chg_4w',
            'prior_shape_enc','days_in_shape']
print(df[key_cols].isnull().sum())

Panel built: 2268 rows
Date range: 2017-01-02 to 2026-06-26

Missing values in key columns:
stock_pct           0
prod_mom_3m        63
oni                 0
usd_myr_chg_4w     20
palm_soy_chg_4w    20
prior_shape_enc     0
days_in_shape       0
dtype: int64


In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report,
    confusion_matrix, accuracy_score)
from sklearn.model_selection import (cross_val_score,
    StratifiedKFold)
from sklearn.feature_selection import RFECV
import matplotlib
matplotlib.use('Agg')

# ── full feature set (all candidates) ─────────────────────────
ALL_FEATURES = {
    # MPOB fundamentals
    'stock_pct':           'Stock Percentile',
    'prod_mom_1m':         'Production Mom 1m',
    'prod_mom_3m':         'Production Mom 3m',
    'prod_yoy':            'Production YoY',
    'export_yoy':          'Export YoY',

    # Macro
    'oni':                 'ENSO ONI',
    'usd_myr_chg_4w':      'USD/MYR Change 4w',
    'palm_soy_chg_4w':     'Palm-Soy Spread 4w',
    'crude_chg_4w':        'Crude Oil Change 4w',

    # Curve history (new signals from validation)
    'days_in_shape':       'Days in Current Shape',
    'prior_shape_enc':     'Prior Shape',
    'month':               'Calendar Month',

    # 2D interaction
    'stock_prod_interaction': 'Stock x Production Quad',
}

# ── backup feature set (current 74.8% model) ──────────────────
BACKUP_FEATURES = [
    'stock_pct', 'prod_mom_1m', 'prod_mom_3m', 'month'
]

feature_cols = [f for f in ALL_FEATURES.keys()
                if f in df.columns]
print(f"Features available: {len(feature_cols)}")
print(f"Features missing: "
      f"{[f for f in ALL_FEATURES if f not in df.columns]}")

# ── train/test split ───────────────────────────────────────────
df_model = df.dropna(subset=feature_cols+['shape']).copy()
df_train = df_model[df_model['date'] <= TRAIN_END]
df_test  = df_model[df_model['date'] >= TEST_START]

X_train = df_train[feature_cols].values
y_train = df_train['shape'].values
X_test  = df_test[feature_cols].values
y_test  = df_test['shape'].values

print(f"\nTrain: {len(df_train)} rows "
      f"({df_train['date'].min().date()} to "
      f"{df_train['date'].max().date()})")
print(f"Test:  {len(df_test)} rows "
      f"({df_test['date'].min().date()} to "
      f"{df_test['date'].max().date()})")
print(f"\nTrain shape distribution:")
for s, n in pd.Series(y_train).value_counts().items():
    print(f"  {display_short(s)}: {n}")

Features available: 13
Features missing: []

Train: 1656 rows (2018-03-22 to 2024-12-31)
Test:  360 rows (2025-01-02 to 2026-06-26)

Train shape distribution:
  SB: 660
  C: 534
  MB: 219
  F: 152
  TB: 91


In [4]:
# ── build backup model on existing feature set ────────────────
# This is the reference. We never replace this.
X_train_bk = df_train[BACKUP_FEATURES].values
X_test_bk  = df_test[BACKUP_FEATURES].values

backup_model = RandomForestClassifier(
    n_estimators=200, max_depth=5,
    min_samples_leaf=10, random_state=42,
    class_weight='balanced')
backup_model.fit(X_train_bk, y_train)

backup_pred   = backup_model.predict(X_test_bk)
backup_acc    = accuracy_score(y_test, backup_pred)*100
backup_cv     = cross_val_score(
    backup_model, X_train_bk, y_train,
    cv=StratifiedKFold(5), scoring='accuracy').mean()*100

print("BACKUP MODEL (74.8% reference)")
print("="*50)
print(f"CV accuracy:   {backup_cv:.1f}%")
print(f"Test accuracy: {backup_acc:.1f}%")
print("\nPer-shape F1 (backup):")
report = classification_report(
    y_test, backup_pred, output_dict=True)
for s in sorted(df['shape'].unique()):
    if s in report:
        f1  = report[s]['f1-score']*100
        prec = report[s]['precision']*100
        rec  = report[s]['recall']*100
        print(f"  {display_short(s)}: "
              f"F1={f1:.0f}%  "
              f"Prec={prec:.0f}%  "
              f"Rec={rec:.0f}%")

output_lines = [
    "BACKUP MODEL (74.8% reference — DO NOT REPLACE)",
    f"CV accuracy:   {backup_cv:.1f}%",
    f"Test accuracy: {backup_acc:.1f}%",
    "\nPer-shape results:"
]
for s in sorted(df['shape'].unique()):
    if s in report:
        output_lines.append(
            f"  {display_shape(s)}: "
            f"F1={report[s]['f1-score']*100:.0f}%  "
            f"Prec={report[s]['precision']*100:.0f}%  "
            f"Rec={report[s]['recall']*100:.0f}%")

write_log("FURTHER STUDY 1 — BACKUP MODEL REFERENCE",
          "\n".join(output_lines))

BACKUP MODEL (74.8% reference)
CV accuracy:   46.8%
Test accuracy: 55.6%

Per-shape F1 (backup):
  SB: F1=56%  Prec=89%  Rec=41%
  MB: F1=0%  Prec=0%  Rec=0%
  TB: F1=21%  Prec=12%  Rec=100%
  C: F1=72%  Prec=61%  Rec=89%
  F: F1=54%  Prec=81%  Rec=41%
[LOG] Written -- FURTHER STUDY 1 — BACKUP MODEL REFERENCE


In [5]:
# ── new model with full variable set ──────────────────────────
new_model = RandomForestClassifier(
    n_estimators=300, max_depth=6,
    min_samples_leaf=8, random_state=42,
    class_weight='balanced')
new_model.fit(X_train, y_train)

new_pred = new_model.predict(X_test)
new_acc  = accuracy_score(y_test, new_pred)*100
new_cv   = cross_val_score(
    new_model, X_train, y_train,
    cv=StratifiedKFold(5),
    scoring='accuracy').mean()*100

print("NEW MODEL (full variable set)")
print("="*50)
print(f"CV accuracy:   {new_cv:.1f}%")
print(f"Test accuracy: {new_acc:.1f}%")
print(f"vs backup:     {new_acc-backup_acc:+.1f}pp")

print("\nPer-shape F1 comparison (backup vs new):")
new_report = classification_report(
    y_test, new_pred, output_dict=True)
output_lines = [
    "NEW MODEL — FULL VARIABLE SET",
    f"CV accuracy:   {new_cv:.1f}%",
    f"Test accuracy: {new_acc:.1f}%",
    f"vs backup:     {new_acc-backup_acc:+.1f}pp",
    "\nPer-shape comparison:"
]
for s in sorted(df['shape'].unique()):
    bk_f1  = report.get(s,{}).get('f1-score',0)*100
    new_f1 = new_report.get(s,{}).get('f1-score',0)*100
    diff   = new_f1 - bk_f1
    flag   = " ***" if diff > 5 else (" !" if diff < -5 else "")
    line   = (f"  {display_shape(s)}: "
              f"backup F1={bk_f1:.0f}%  "
              f"new F1={new_f1:.0f}%  "
              f"diff={diff:+.0f}pp{flag}")
    print(line)
    output_lines.append(line)

# Feature importance
print("\nFeature importance (new model):")
fi = sorted(zip(feature_cols,
                new_model.feature_importances_),
            key=lambda x: -x[1])
output_lines.append("\nFeature importance:")
for feat, imp in fi:
    line = (f"  {ALL_FEATURES.get(feat,feat):<35}"
            f"{imp:.3f}")
    print(line)
    output_lines.append(line)

write_log("FURTHER STUDY 1 — NEW MODEL (FULL VARIABLES)",
          "\n".join(output_lines))

NEW MODEL (full variable set)
CV accuracy:   84.7%
Test accuracy: 80.8%
vs backup:     +25.3pp

Per-shape F1 comparison (backup vs new):
  SB  (Steep Backwardation): backup F1=56%  new F1=90%  diff=+34pp ***
  MB  (Mild Backwardation): backup F1=0%  new F1=27%  diff=+27pp ***
  TB  (Transitional Backwardation): backup F1=21%  new F1=36%  diff=+15pp ***
  C   (Contango): backup F1=72%  new F1=83%  diff=+11pp ***
  F   (Flat/Back-Inversion): backup F1=54%  new F1=84%  diff=+30pp ***

Feature importance (new model):
  Prior Shape                        0.245
  Stock Percentile                   0.201
  Production Mom 3m                  0.114
  Days in Current Shape              0.106
  ENSO ONI                           0.066
  Stock x Production Quad            0.060
  Palm-Soy Spread 4w                 0.045
  Production Mom 1m                  0.044
  Calendar Month                     0.041
  Production YoY                     0.025
  USD/MYR Change 4w                  0.021
  Export

In [6]:
# ── recursive feature elimination ─────────────────────────────
# Finds the optimal subset of variables that maximises
# cross-validated accuracy — removes variables that
# add noise rather than signal

from sklearn.feature_selection import RFECV

print("Running variable optimization...")
print("(This may take a few minutes)")

selector = RFECV(
    estimator=RandomForestClassifier(
        n_estimators=100, max_depth=5,
        min_samples_leaf=10, random_state=42,
        class_weight='balanced'),
    step=1,
    cv=StratifiedKFold(5),
    scoring='accuracy',
    min_features_to_select=3,
    n_jobs=-1)

selector.fit(X_train, y_train)

optimal_features = [feature_cols[i]
                    for i in range(len(feature_cols))
                    if selector.support_[i]]

print(f"\nOptimal feature count: {selector.n_features_}")
print(f"Best CV score: {selector.cv_results_['mean_test_score'].max()*100:.1f}%")
print(f"\nSelected features:")
for f in optimal_features:
    print(f"  {ALL_FEATURES.get(f,f)}")

print(f"\nRemoved features:")
removed = [f for f in feature_cols
           if f not in optimal_features]
for f in removed:
    print(f"  {ALL_FEATURES.get(f,f)} — added noise, removed")

output_lines = [
    "VARIABLE OPTIMIZATION — RFECV RESULTS",
    f"Optimal feature count: {selector.n_features_}",
    f"Best CV score: "
    f"{selector.cv_results_['mean_test_score'].max()*100:.1f}%",
    "\nSelected features:"
]
for f in optimal_features:
    output_lines.append(f"  {ALL_FEATURES.get(f,f)}")
output_lines.append("\nRemoved features:")
for f in removed:
    output_lines.append(
        f"  {ALL_FEATURES.get(f,f)} — added noise, removed")

write_log("FURTHER STUDY 1 — VARIABLE OPTIMIZATION",
          "\n".join(output_lines))

Running variable optimization...
(This may take a few minutes)



Optimal feature count: 11
Best CV score: 86.9%

Selected features:
  Stock Percentile
  Production Mom 1m
  Production Mom 3m
  Production YoY
  ENSO ONI
  USD/MYR Change 4w
  Palm-Soy Spread 4w
  Days in Current Shape
  Prior Shape
  Calendar Month
  Stock x Production Quad

Removed features:
  Export YoY — added noise, removed
  Crude Oil Change 4w — added noise, removed
[LOG] Written -- FURTHER STUDY 1 — VARIABLE OPTIMIZATION


In [7]:
# ── build final optimized model ───────────────────────────────
X_train_opt = df_train[optimal_features].values
X_test_opt  = df_test[optimal_features].values

opt_model = RandomForestClassifier(
    n_estimators=300, max_depth=6,
    min_samples_leaf=8, random_state=42,
    class_weight='balanced')
opt_model.fit(X_train_opt, y_train)

opt_pred = opt_model.predict(X_test_opt)
opt_acc  = accuracy_score(y_test, opt_pred)*100
opt_cv   = cross_val_score(
    opt_model, X_train_opt, y_train,
    cv=StratifiedKFold(5),
    scoring='accuracy').mean()*100

print("OPTIMIZED MODEL (best variable subset)")
print("="*60)
print(f"\n{'Model':<20} {'CV Acc':>8} {'Test Acc':>10} "
      f"{'vs Backup':>10}")
print("-"*50)
print(f"{'Backup (74.8%)':<20} {backup_cv:>7.1f}% "
      f"{backup_acc:>9.1f}% {'—':>10}")
print(f"{'Full variables':<20} {new_cv:>7.1f}% "
      f"{new_acc:>9.1f}% {new_acc-backup_acc:>+9.1f}pp")
print(f"{'Optimized subset':<20} {opt_cv:>7.1f}% "
      f"{opt_acc:>9.1f}% {opt_acc-backup_acc:>+9.1f}pp")

print("\nPer-shape F1 — all three models:")
opt_report = classification_report(
    y_test, opt_pred, output_dict=True)

output_lines = [
    "OPTIMIZED MODEL — FINAL COMPARISON",
    f"{'Model':<20} {'CV':>8} {'Test':>8} {'vs Backup':>10}",
    f"Backup:           {backup_cv:.1f}%   "
    f"{backup_acc:.1f}%   —",
    f"Full variables:   {new_cv:.1f}%   "
    f"{new_acc:.1f}%   {new_acc-backup_acc:+.1f}pp",
    f"Optimized subset: {opt_cv:.1f}%   "
    f"{opt_acc:.1f}%   {opt_acc-backup_acc:+.1f}pp",
    "\nPer-shape F1:"
]

for s in sorted(df['shape'].unique()):
    bk  = report.get(s,{}).get('f1-score',0)*100
    new = new_report.get(s,{}).get('f1-score',0)*100
    opt = opt_report.get(s,{}).get('f1-score',0)*100
    line = (f"  {display_shape(s)}: "
            f"backup={bk:.0f}%  "
            f"full={new:.0f}%  "
            f"optimized={opt:.0f}%")
    print(line)
    output_lines.append(line)

# Verdict
best_acc = max(backup_acc, new_acc, opt_acc)
best_name = (
    "Optimized" if opt_acc == best_acc else
    "Full variables" if new_acc == best_acc
    else "Backup")

output_lines.append(
    f"\nVERDICT: Best model = {best_name} "
    f"at {best_acc:.1f}% test accuracy")
print(f"\nVERDICT: Best model = {best_name} "
      f"at {best_acc:.1f}% test accuracy")

write_log("FURTHER STUDY 1 — OPTIMIZED MODEL FINAL",
          "\n".join(output_lines))
print("\n[NOTEBOOK 1 COMPLETE]")

OPTIMIZED MODEL (best variable subset)

Model                  CV Acc   Test Acc  vs Backup
--------------------------------------------------
Backup (74.8%)          46.8%      55.6%          —
Full variables          84.7%      80.8%     +25.3pp
Optimized subset        85.0%      84.7%     +29.2pp

Per-shape F1 — all three models:
  SB  (Steep Backwardation): backup=56%  full=90%  optimized=92%
  MB  (Mild Backwardation): backup=0%  full=27%  optimized=26%
  TB  (Transitional Backwardation): backup=21%  full=36%  optimized=32%
  C   (Contango): backup=72%  full=83%  optimized=87%
  F   (Flat/Back-Inversion): backup=54%  full=84%  optimized=91%

VERDICT: Best model = Optimized at 84.7% test accuracy
[LOG] Written -- FURTHER STUDY 1 — OPTIMIZED MODEL FINAL

[NOTEBOOK 1 COMPLETE]


In [8]:
# ── Year-by-year accuracy across FULL dataset ─────────────────
# Training years: in-sample (will be higher — expected)
# Test years: out-of-sample (the honest numbers)
# Purpose: check regime coverage, not just overall accuracy

df_full_check = df_model.copy()
df_full_check['year'] = df_full_check['date'].dt.year
df_full_check['split'] = np.where(
    df_full_check['date'] <= TRAIN_END,
    'TRAIN', 'TEST')

# Predict on full dataset
# Training = in-sample (optimistic, expected)
# Test = out-of-sample (honest)
X_full     = df_full_check[optimal_features].values
y_full     = df_full_check['shape'].values
pred_full  = opt_model.predict(X_full)
df_full_check['predicted'] = pred_full
df_full_check['correct']   = (pred_full == y_full)

print("YEAR-BY-YEAR ACCURACY — OPTIMIZED MODEL")
print("(TRAIN = in-sample, TEST = out-of-sample)")
print("="*70)
print(f"\n{'Year':<6} {'Split':>6} {'Accuracy':>10} "
      f"{'n':>6} {'Dominant Shape':>20} "
      f"{'Dom %':>7} {'Note'}")
print("-"*70)

output_lines = [
    "YEAR-BY-YEAR ACCURACY — FULL DATASET CHECK",
    "="*70,
    f"{'Year':<6} {'Split':>6} {'Accuracy':>10} "
    f"{'n':>6} {'Dominant Shape':>20} {'Dom%':>7}"
]

prev_split = None
for year, grp in df_full_check.groupby('year'):
    acc      = grp['correct'].mean()*100
    n        = len(grp)
    split    = grp['split'].iloc[0]
    dom_s    = grp['shape'].value_counts().index[0]
    dom_pct  = grp['shape'].value_counts().iloc[0]/n*100

    # Flag years where model struggles
    flag = ""
    if split == 'TEST' and acc < 75:
        flag = " *** WEAK ON TEST"
    elif split == 'TRAIN' and acc < 70:
        flag = " *** WEAK IN-SAMPLE"
    elif split == 'TEST' and acc > 90:
        flag = " (may be regime-specific)"

    # Add separator between train and test
    if prev_split == 'TRAIN' and split == 'TEST':
        print("  " + "─"*65 + " TEST SET BELOW")
        output_lines.append("─"*65 + " TEST SET BELOW")

    line = (f"  {year:<4} {split:>6}  "
            f"{acc:>8.1f}%  "
            f"{n:>6}  "
            f"{display_short(dom_s):>20}  "
            f"{dom_pct:>5.0f}%  "
            f"{flag}")
    print(line)
    output_lines.append(line)
    prev_split = split

# Per-shape accuracy by year — the deeper check
print("\n\nPER-SHAPE ACCURACY BY YEAR")
print("(how well does the model identify each shape "
      "in each year?)")
print("="*70)
output_lines.append("\nPER-SHAPE ACCURACY BY YEAR")
output_lines.append("="*70)

shape_year = df_full_check.groupby(
    ['year','shape']).apply(
    lambda x: pd.Series({
        'n':        len(x),
        'accuracy': (x['correct'].mean()*100).round(1),
        'split':    x['split'].iloc[0]
    })).reset_index()

for s in sorted(df_full_check['shape'].unique()):
    sub = shape_year[shape_year['shape']==s]
    line = f"\n  {display_shape(s)}:"
    print(line); output_lines.append(line)

    header = (f"  {'Year':<6} {'Split':>6} "
              f"{'Accuracy':>10} {'n':>6}")
    print(header); output_lines.append(header)

    for _, row in sub.iterrows():
        flag = ""
        if row['split']=='TEST' and row['accuracy'] < 60:
            flag = " *** WEAK"
        elif row['n'] < 10:
            flag = " (low n)"
        line = (f"    {int(row['year']):<4} "
                f"{row['split']:>6}  "
                f"{row['accuracy']:>8.1f}%  "
                f"{int(row['n']):>6}"
                f"{flag}")
        print(line); output_lines.append(line)

# Overall consistency check
print("\n\nCONSISTENCY SUMMARY")
print("="*40)
output_lines.append("\nCONSISTENCY SUMMARY")
output_lines.append("="*40)

test_by_year = df_full_check[
    df_full_check['split']=='TEST'].groupby('year')[
    'correct'].mean()*100
train_by_year = df_full_check[
    df_full_check['split']=='TRAIN'].groupby('year')[
    'correct'].mean()*100

print(f"\nTest years accuracy range: "
      f"{test_by_year.min():.1f}% to "
      f"{test_by_year.max():.1f}%")
print(f"Train years accuracy range: "
      f"{train_by_year.min():.1f}% to "
      f"{train_by_year.max():.1f}%")

gap = train_by_year.mean() - test_by_year.mean()
print(f"\nAverage train-test gap: {gap:.1f}pp")
if gap > 15:
    verdict = ("LARGE GAP — model may be overfitting. "
               "Treat 84.7% with caution.")
elif gap > 8:
    verdict = ("MODERATE GAP — some overfitting possible. "
               "84.7% is slightly optimistic.")
else:
    verdict = ("SMALL GAP — model generalises well. "
               "84.7% is a trustworthy number.")

print(f"Verdict: {verdict}")
output_lines.append(
    f"\nTrain accuracy: {train_by_year.mean():.1f}%")
output_lines.append(
    f"Test accuracy:  {test_by_year.mean():.1f}%")
output_lines.append(f"Gap: {gap:.1f}pp")
output_lines.append(f"Verdict: {verdict}")

write_log("FURTHER STUDY 1 — YEAR BY YEAR ACCURACY CHECK",
          "\n".join(output_lines))
print("\n[YEAR-BY-YEAR CHECK COMPLETE]")

YEAR-BY-YEAR ACCURACY — OPTIMIZED MODEL
(TRAIN = in-sample, TEST = out-of-sample)

Year    Split   Accuracy      n       Dominant Shape   Dom % Note
----------------------------------------------------------------------
  2018  TRAIN      97.8%     185                     C     85%  
  2019  TRAIN      99.6%     244                     C     83%  
  2020  TRAIN      96.0%     248                    SB     69%  
  2021  TRAIN      99.2%     245                    SB     92%  
  2022  TRAIN      91.8%     243                    SB     36%  
  2023  TRAIN      94.3%     244                     C     37%  
  2024  TRAIN      93.5%     247                    SB     51%  
  ───────────────────────────────────────────────────────────────── TEST SET BELOW
  2025   TEST      84.0%     244                    SB     33%  
  2026   TEST      86.2%     116                     F     54%  


PER-SHAPE ACCURACY BY YEAR
(how well does the model identify each shape in each year?)

  SB  (Steep Backwarda